<a href="https://colab.research.google.com/github/CoderMakar/Plenki/blob/main/MOST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# DATABASE A — NBD/QC MOST
# Чистый полный pipeline для Google Colab
# ============================================================

import os
import sys
import subprocess
import zipfile
import warnings

warnings.filterwarnings("ignore")


# ============================================================
# 1. УСТАНОВКА ЗАВИСИМОСТЕЙ
# ============================================================

print("=== 1. Установка зависимостей ===")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "rdkit"],
    check=True
)

# xyz2mol нужен старому авторскому коду qmconf
XYZ2MOL_DIR = "/content/xyz2mol"

if not os.path.isdir(XYZ2MOL_DIR):
    subprocess.run(
        [
            "git",
            "clone",
            "-q",
            "https://github.com/jensengroup/xyz2mol.git",
            XYZ2MOL_DIR
        ],
        check=True
    )

print("Зависимости готовы.")


# ============================================================
# 2. СКАЧИВАНИЕ АРХИВА
# ============================================================

print("\n=== 2. Исходный архив ===")

ZIP_PATH = "/content/project.zip"

URL = (
    "https://sid.erda.dk/share_redirect/"
    "DeRV97z1Nz/"
    "virtual_screening_of_norbornadiene_based_"
    "molecular_solar_thermal_energy_storage_systems_"
    "using_a_genetic_algorithm.zip"
)

if not os.path.isfile(ZIP_PATH):

    print("Скачиваю архив...")

    subprocess.run(
        [
            "wget",
            "-q",
            "--show-progress",
            URL,
            "-O",
            ZIP_PATH
        ],
        check=True
    )

else:
    print("Архив уже загружен:", ZIP_PATH)


# ============================================================
# 3. РАСПАКОВКА
# ============================================================

print("\n=== 3. Распаковка ===")

EXTRACT_DIR = "/content/project"

if not os.path.isdir(EXTRACT_DIR):
    os.makedirs(EXTRACT_DIR, exist_ok=True)

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_DIR)

    print("Архив распакован.")
else:
    print("Папка проекта уже существует.")


# ============================================================
# 4. ПУТИ ПРОЕКТА
# ============================================================

BASE = os.path.join(
    EXTRACT_DIR,
    "virtual_screening_of_norbornadiene_based_"
    "molecular_solar_thermal_energy_storage_systems_"
    "using_a_genetic_algorithm"
)

QMC = os.path.join(
    BASE,
    "dependencies",
    "tQMC",
    "QMC"
)

CALC_DIR = os.path.join(
    QMC,
    "calculator"
)

DATASET_DIR = os.path.join(
    BASE,
    "dataset"
)

RAW_PATH = os.path.join(
    DATASET_DIR,
    "result_abs.pkl"
)

READY_PATH = os.path.join(
    DATASET_DIR,
    "result_abs_700_nodublicates.pkl"
)


required_paths = {
    "QMC": QMC,
    "calculator": CALC_DIR,
    "RAW dataset": RAW_PATH,
    "READY dataset": READY_PATH
}

for name, path in required_paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{name} не найден:\n{path}"
        )

print("\nВсе исходные файлы найдены.")


# ============================================================
# 5. ПОДКЛЮЧЕНИЕ АВТОРСКИХ МОДУЛЕЙ
# ============================================================

print("\n=== 4. Подключение QMC ===")

# calculator должен быть Python-пакетом
init_file = os.path.join(
    CALC_DIR,
    "__init__.py"
)

if not os.path.exists(init_file):
    with open(init_file, "w"):
        pass


# Нельзя добавлять QMC/calculator напрямую,
# иначе calculator импортируется как calculator.py,
# а не как пакет calculator.*
sys.path = [
    p for p in sys.path
    if p != CALC_DIR
]

if QMC not in sys.path:
    sys.path.insert(0, QMC)

if "/content" not in sys.path:
    sys.path.insert(0, "/content")


# Очищаем возможные старые импорты
for module_name in list(sys.modules):

    if (
        module_name == "qmconf"
        or module_name == "calculator"
        or module_name.startswith("calculator.")
    ):
        del sys.modules[module_name]


from qmconf import QMConf
import calculator.xtb

print("QMC подключён.")
print("calculator.xtb:", calculator.xtb.__file__)


# ============================================================
# 6. ЗАГРУЗКА DATASET
# ============================================================

print("\n=== 5. Загрузка данных ===")

import pandas as pd
import numpy as np

df_raw = pd.read_pickle(
    RAW_PATH
)

df_ready = pd.read_pickle(
    READY_PATH
)


print(
    "RAW:",
    df_raw.shape
)

print(
    "READY:",
    df_ready.shape
)

print(
    "READY columns:",
    df_ready.columns.tolist()
)


# Проверки готовой авторской выборки
assert len(df_ready) == 33706
assert df_ready["smiles"].nunique() == 33706
assert df_ready.isna().sum().sum() == 0

print(
    "READY dataset корректен:",
    "33706 уникальных записей."
)


# ============================================================
# 7. СОЗДАНИЕ comp_name В RAW
# ============================================================

print("\n=== 6. Связь READY ↔ RAW ===")

df_raw = df_raw.copy()

df_raw["comp_name"] = (
    df_raw["prod"]
    .apply(
        lambda obj:
        str(obj.label.split("_p")[0])
    )
)

df_raw["raw_index"] = df_raw.index


print(
    "RAW строк:",
    len(df_raw)
)

print(
    "RAW уникальных comp_name:",
    df_raw["comp_name"].nunique()
)


assert df_raw["comp_name"].is_unique


# ============================================================
# 8. MERGE ПО comp_name
# ============================================================

raw_map = df_raw[
    [
        "raw_index",
        "comp_name",
        "reac",
        "prod"
    ]
].copy()


ready_map = df_ready[
    [
        "comp_name",
        "smiles",
        "storage",
        "tbr",
        "absorp"
    ]
].copy()


ready_map = ready_map.rename(
    columns={
        "smiles": "qc_smiles_raw"
    }
)


matched = ready_map.merge(
    raw_map,
    on="comp_name",
    how="left",
    validate="one_to_one"
)


print(
    "Сопоставлено:",
    matched["raw_index"].notna().sum(),
    "/",
    len(matched)
)


assert matched["raw_index"].notna().all()


# ============================================================
# 9. ИЗВЛЕЧЕНИЕ NBD И QC SMILES
# ============================================================

print("\n=== 7. Подготовка структур ===")

from rdkit import Chem
from rdkit.Chem import Descriptors


def qmconf_to_smiles(obj):
    """
    QMConf -> canonical SMILES.
    Используется для исходного NBD (reac).
    """

    try:

        mol = obj.rdkit_mol

        if mol is None:
            mol = obj.get_rdkit_mol()

        if mol is None:
            return None

        mol = Chem.RemoveHs(mol)

        return Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=False
        )

    except Exception:
        return None


def normalize_smiles(smiles):
    """
    SMILES -> canonical SMILES.
    """

    try:

        mol = Chem.MolFromSmiles(
            smiles
        )

        if mol is None:
            return None

        mol = Chem.RemoveHs(mol)

        return Chem.MolToSmiles(
            mol,
            canonical=True,
            isomericSmiles=False
        )

    except Exception:
        return None


# NBD из RAW reac
matched["nbd_smiles"] = (
    matched["reac"]
    .apply(qmconf_to_smiles)
)


# QC из GCN-ready набора авторов
matched["qc_smiles"] = (
    matched["qc_smiles_raw"]
    .apply(normalize_smiles)
)


print(
    "NBD получено:",
    matched["nbd_smiles"].notna().sum()
)

print(
    "QC получено:",
    matched["qc_smiles"].notna().sum()
)

print(
    "Уникальных NBD:",
    matched["nbd_smiles"].nunique()
)

print(
    "Уникальных QC:",
    matched["qc_smiles"].nunique()
)


assert matched["nbd_smiles"].notna().all()
assert matched["qc_smiles"].notna().all()

assert (
    matched["nbd_smiles"].nunique()
    == 33706
)

assert (
    matched["qc_smiles"].nunique()
    == 33706
)


# ============================================================
# 10. МОЛЯРНАЯ МАССА
# ============================================================

print("\n=== 8. Расчёт энергетической плотности ===")


def get_mol_weight(smiles):

    mol = Chem.MolFromSmiles(
        smiles
    )

    if mol is None:
        return np.nan

    return Descriptors.MolWt(
        mol
    )


matched["mol_weight_g_mol"] = (
    matched["nbd_smiles"]
    .apply(get_mol_weight)
)


# ============================================================
# 11. ENERGY DENSITY
#
# storage          kJ/mol
# molecular weight g/mol
#
# kJ/mol ÷ g/mol
# = kJ/g
# = MJ/kg
# ============================================================

matched["energy_density_MJ_kg"] = (
    matched["storage"]
    /
    matched["mol_weight_g_mol"]
)


# ============================================================
# 12. DATABASE A
# ============================================================

print("\n=== 9. Формирование Database A ===")


database_A_final = pd.DataFrame({

    "comp_name":
        matched["comp_name"],

    "nbd_smiles":
        matched["nbd_smiles"],

    "qc_smiles":
        matched["qc_smiles"],

    "storage_kJ_mol":
        matched["storage"],

    "energy_density_MJ_kg":
        matched["energy_density_MJ_kg"],

    "tbr_kJ_mol":
        matched["tbr"],

    "absorption_nm":
        matched["absorp"]
})


# ============================================================
# 13. ПРОВЕРКА DATABASE A
# ============================================================

print(
    "Размер:",
    database_A_final.shape
)

print(
    "Уникальных NBD:",
    database_A_final[
        "nbd_smiles"
    ].nunique()
)

print(
    "Уникальных QC:",
    database_A_final[
        "qc_smiles"
    ].nunique()
)

print("\nПропуски:")

print(
    database_A_final
    .isna()
    .sum()
)


assert database_A_final.shape == (
    33706,
    7
)

assert (
    database_A_final
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    database_A_final[
        "nbd_smiles"
    ].is_unique
)

assert (
    database_A_final[
        "qc_smiles"
    ].is_unique
)


# ============================================================
# 14. СТАТИСТИКА
# ============================================================

print(
    "\n=== Статистика свойств ==="
)

display(

    database_A_final[
        [
            "storage_kJ_mol",
            "energy_density_MJ_kg",
            "tbr_kJ_mol",
            "absorption_nm"
        ]
    ]
    .describe()

)


print(
    "\n=== Первые молекулы ==="
)

display(
    database_A_final.head(10)
)


# ============================================================
# 15. DATASET ДЛЯ ГЕНЕРАТОРА
# ============================================================

generator_corpus = (

    database_A_final[
        [
            "comp_name",
            "nbd_smiles"
        ]
    ]

    .rename(
        columns={
            "nbd_smiles":
            "smiles"
        }
    )

    .copy()
)


# ============================================================
# 16. DATASET ДЛЯ MOST EXPERT
# ============================================================

most_expert_data = (

    database_A_final[
        [
            "comp_name",
            "nbd_smiles",
            "energy_density_MJ_kg",
            "tbr_kJ_mol"
        ]
    ]

    .rename(
        columns={
            "nbd_smiles":
            "smiles"
        }
    )

    .copy()
)


print(
    "\nGenerator:",
    generator_corpus.shape
)

print(
    "MOST expert:",
    most_expert_data.shape
)


# ============================================================
# 17. СОХРАНЕНИЕ
# ============================================================

OUT_A = (
    "/content/"
    "database_A_MOST_final.csv"
)

OUT_GENERATOR = (
    "/content/"
    "NBD_generator_corpus.csv"
)

OUT_EXPERT = (
    "/content/"
    "MOST_expert_dataset.csv"
)


database_A_final.to_csv(
    OUT_A,
    index=False
)

generator_corpus.to_csv(
    OUT_GENERATOR,
    index=False
)

most_expert_data.to_csv(
    OUT_EXPERT,
    index=False
)


# ============================================================
# 18. ИТОГ
# ============================================================

print(
    "\n======================================"
)

print(
    "DATABASE A ГОТОВА"
)

print(
    "======================================"
)

print(
    "Database A:",
    database_A_final.shape
)

print(
    "Generator:",
    generator_corpus.shape
)

print(
    "MOST expert:",
    most_expert_data.shape
)

print(
    "\nСохранено:"
)

print(
    OUT_A
)

print(
    OUT_GENERATOR
)

print(
    OUT_EXPERT
)

=== 1. Установка зависимостей ===
Зависимости готовы.

=== 2. Исходный архив ===
Скачиваю архив...

=== 3. Распаковка ===
Архив распакован.

Все исходные файлы найдены.

=== 4. Подключение QMC ===
QMC подключён.
calculator.xtb: /content/project/virtual_screening_of_norbornadiene_based_molecular_solar_thermal_energy_storage_systems_using_a_genetic_algorithm/dependencies/tQMC/QMC/calculator/xtb.py

=== 5. Загрузка данных ===
RAW: (55568, 7)
READY: (33706, 5)
READY columns: ['comp_name', 'smiles', 'storage', 'tbr', 'absorp']
READY dataset корректен: 33706 уникальных записей.

=== 6. Связь READY ↔ RAW ===
RAW строк: 55568
RAW уникальных comp_name: 55568
Сопоставлено: 33706 / 33706

=== 7. Подготовка структур ===
NBD получено: 33706
QC получено: 33706
Уникальных NBD: 33706
Уникальных QC: 33706

=== 8. Расчёт энергетической плотности ===

=== 9. Формирование Database A ===
Размер: (33706, 7)
Уникальных NBD: 33706
Уникальных QC: 33706

Пропуски:
comp_name               0
nbd_smiles           

,storage_kJ_mol,energy_density_MJ_kg,tbr_kJ_mol,absorption_nm
count,33706.000000,33706.000000,33706.000000,33706.000000
mean,23.961666,0.079843,212.928727,325.045559
std,8.584163,0.035754,31.323953,41.013722
min,0.099234,0.000310,65.437325,159.700000
25%,17.236745,0.060036,192.306838,297.600000
50%,23.369080,0.073757,210.299817,314.600000
75%,28.854381,0.088511,242.486067,344.400000
max,86.068099,0.569707,320.298085,643.500000



=== Первые молекулы ===


,comp_name,nbd_smiles,qc_smiles,storage_kJ_mol,energy_density_MJ_kg,tbr_kJ_mol,absorption_nm
0,A-0_A-0_A-0_A-0,C1=CC2C=CC1C2,C1C2C3C2C2C1C32,8.111205,0.088030,320.298085,159.7
1,A-1_A-0_A-0_A-0,FC1=CC2C=CC1C2,FC12C3CC4C(C41)C32,22.362574,0.203054,290.434249,177.7
2,A-2_A-0_A-0_A-0,FC(F)(F)C1=CC2C=CC1C2,FC(F)(F)C12C3CC4C(C41)C32,7.536407,0.047062,284.936736,167.8
3,A-3_A-0_A-0_A-0,N#CC1=CC2C=CC1C2,N#CC12C3CC4C(C41)C32,9.834636,0.083948,258.947101,220.8
4,A-4_A-0_A-0_A-0,O=[N+]([O-])C1=CC2C=CC1C2,O=[N+]([O-])C12C3CC4C(C41)C32,5.430744,0.039601,219.128675,295.2
5,A-5_A-0_A-0_A-0,O=CC1=CC2C=CC1C2,O=CC12C3CC4C(C41)C32,3.724758,0.031001,240.969550,239.4
6,A-6_A-0_A-0_A-0,O=C(O)C1=CC2C=CC1C2,O=C(O)C12C3CC4C(C41)C32,3.173696,0.023310,252.096535,232.3
7,A-7_A-0_A-0_A-0,CC(=O)C1=CC2C=CC1C2,CC(=O)C12C3CC4C(C41)C32,2.787060,0.020771,248.139052,294.6
8,A-8_A-0_A-0_A-0,NC(=O)C1=CC2C=CC1C2,NC(=O)C12C3CC4C(C41)C32,5.610543,0.041509,260.607473,238.2
9,A-9_A-0_A-0_A-0,CS(=O)(=O)C1=CC2C=CC1C2,CS(=O)(=O)C12C3CC4C(C41)C32,5.274700,0.030985,282.684282,230.3



Generator: (33706, 2)
MOST expert: (33706, 4)

DATABASE A ГОТОВА
Database A: (33706, 7)
Generator: (33706, 2)
MOST expert: (33706, 4)

Сохранено:
/content/database_A_MOST_final.csv
/content/NBD_generator_corpus.csv
/content/MOST_expert_dataset.csv


In [3]:
# ============================================================
# DATABASE A / MOST
# SPLIT + PROPERTY BASELINE + EXPERT A
#
# Вход:
#   /content/MOST_expert_dataset.csv
#
# Ожидаемые колонки:
#   comp_name
#   smiles
#   energy_density_MJ_kg
#   tbr_kJ_mol
#
# Выход:
#   /content/MOST_expert_A/
# ============================================================

import os
import json
import time
import joblib
import warnings

import numpy as np
import pandas as pd

from rdkit import Chem, DataStructs
from rdkit.Chem import (
    Descriptors,
    Crippen,
    Lipinski,
    rdMolDescriptors,
    rdFingerprintGenerator
)
from rdkit.Chem.Scaffolds import MurckoScaffold

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

# ============================================================
# 0. CONFIG
# ============================================================

SEED = 42

DATA_PATH = "/content/MOST_expert_dataset.csv"
OUTPUT_DIR = "/content/MOST_expert_A"

os.makedirs(OUTPUT_DIR, exist_ok=True)

FP_RADIUS = 2
FP_SIZE = 1024

TRAIN_FRAC = 0.80
VAL_FRAC = 0.10
TEST_FRAC = 0.10

# Для applicability domain не нужно сравнивать
# каждую молекулу со всеми 27k train-молекулами.
# Берём фиксированную воспроизводимую reference subset.
AD_REFERENCE_SIZE = 5000

TARGETS = [
    "energy_density_MJ_kg",
    "tbr_kJ_mol"
]

np.random.seed(SEED)


# ============================================================
# 1. LOAD DATA
# ============================================================

print("=" * 70)
print("1. LOAD DATABASE A")
print("=" * 70)

data = pd.read_csv(DATA_PATH)

required = [
    "comp_name",
    "smiles",
    "energy_density_MJ_kg",
    "tbr_kJ_mol"
]

missing = [
    col for col in required
    if col not in data.columns
]

if missing:
    raise ValueError(
        f"Нет необходимых колонок: {missing}"
    )

if data[required].isna().sum().sum() != 0:
    raise ValueError(
        "В данных обнаружены пропуски."
    )

if data["smiles"].duplicated().any():
    raise ValueError(
        "Обнаружены дубли SMILES."
    )

print("Размер:", data.shape)
print("Уникальных SMILES:", data["smiles"].nunique())

display(data.head())


# ============================================================
# 2. VALIDATE SMILES
# ============================================================

print("\n" + "=" * 70)
print("2. SMILES VALIDATION")
print("=" * 70)

mols = []

invalid = []

for i, smi in enumerate(data["smiles"]):

    mol = Chem.MolFromSmiles(smi)

    if mol is None:
        invalid.append(i)

    mols.append(mol)

print("Валидных:", len(data) - len(invalid))
print("Невалидных:", len(invalid))

if invalid:
    raise ValueError(
        f"Есть invalid SMILES: {invalid[:10]}"
    )


# ============================================================
# 3. BEMIS-MURCKO SCAFFOLDS
#
# Сначала проверяем, пригоден ли scaffold split
# для этой конкретной NBD-базы.
# ============================================================

print("\n" + "=" * 70)
print("3. SCAFFOLD DIAGNOSTICS")
print("=" * 70)


def get_scaffold(smiles):

    return MurckoScaffold.MurckoScaffoldSmiles(
        smiles=smiles,
        includeChirality=False
    )


data["scaffold"] = (
    data["smiles"]
    .apply(get_scaffold)
)

scaffold_counts = (
    data["scaffold"]
    .value_counts()
)

n_scaffolds = len(scaffold_counts)

largest_scaffold = int(
    scaffold_counts.iloc[0]
)

largest_fraction = (
    largest_scaffold / len(data)
)

print("Уникальных scaffold:", n_scaffolds)

print(
    "Самый большой scaffold:",
    largest_scaffold,
    "молекул"
)

print(
    "Доля крупнейшего scaffold:",
    round(largest_fraction, 4)
)


# ============================================================
# 4. SPLIT STRATEGY
#
# Если Murcko scaffold split разумен —
# используем его.
#
# Если почти вся NBD-база имеет один scaffold,
# scaffold split теряет смысл, поэтому используем
# воспроизводимый random split с балансировкой
# распределений targets.
# ============================================================

USE_SCAFFOLD_SPLIT = (
    n_scaffolds >= 10
    and largest_fraction < 0.70
)

print(
    "\nВыбранный split:",
    "scaffold"
    if USE_SCAFFOLD_SPLIT
    else "stratified random"
)


# ============================================================
# 5A. SCAFFOLD SPLIT
# ============================================================

def best_group_split(
    indices,
    groups,
    target_test_fraction,
    seed,
    n_trials=100
):

    indices = np.asarray(indices)

    groups = np.asarray(groups)

    best = None
    best_error = float("inf")

    splitter = GroupShuffleSplit(
        n_splits=n_trials,
        test_size=target_test_fraction,
        random_state=seed
    )

    dummy = np.zeros(
        len(indices)
    )

    for train_local, test_local in splitter.split(
        dummy,
        groups=groups
    ):

        actual_fraction = (
            len(test_local) /
            len(indices)
        )

        error = abs(
            actual_fraction -
            target_test_fraction
        )

        if error < best_error:

            best_error = error

            best = (
                indices[train_local],
                indices[test_local]
            )

    return best


if USE_SCAFFOLD_SPLIT:

    all_indices = np.arange(
        len(data)
    )

    train_idx, temp_idx = best_group_split(
        indices=all_indices,
        groups=data["scaffold"].values,
        target_test_fraction=0.20,
        seed=SEED
    )

    # Validation/test 50:50 внутри оставшихся 20%
    temp_groups = (
        data.iloc[temp_idx]["scaffold"]
        .values
    )

    val_idx, test_idx = best_group_split(
        indices=temp_idx,
        groups=temp_groups,
        target_test_fraction=0.50,
        seed=SEED + 1
    )


# ============================================================
# 5B. STRATIFIED RANDOM SPLIT
#
# Для regression делаем приблизительную стратификацию:
# делим оба target на квартили и объединяем bins.
# ============================================================

else:

    energy_bins = pd.qcut(
        data["energy_density_MJ_kg"],
        q=4,
        labels=False,
        duplicates="drop"
    )

    tbr_bins = pd.qcut(
        data["tbr_kJ_mol"],
        q=4,
        labels=False,
        duplicates="drop"
    )

    strata = (
        energy_bins.astype(str)
        + "_"
        + tbr_bins.astype(str)
    )

    all_indices = np.arange(
        len(data)
    )

    train_idx, temp_idx = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=SEED,
        stratify=strata
    )

    temp_strata = strata.iloc[
        temp_idx
    ]

    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=0.50,
        random_state=SEED + 1,
        stratify=temp_strata
    )


train_idx = np.asarray(train_idx)
val_idx = np.asarray(val_idx)
test_idx = np.asarray(test_idx)


# ============================================================
# 6. SPLIT CHECK
# ============================================================

print("\n" + "=" * 70)
print("4. SPLIT RESULT")
print("=" * 70)

print(
    "Train:",
    len(train_idx),
    f"({len(train_idx)/len(data):.3f})"
)

print(
    "Validation:",
    len(val_idx),
    f"({len(val_idx)/len(data):.3f})"
)

print(
    "Test:",
    len(test_idx),
    f"({len(test_idx)/len(data):.3f})"
)

if len(train_idx) > 30000:

    raise RuntimeError(
        "Train > 30 000 молекул."
    )


# Проверка пересечения индексов

assert len(
    set(train_idx)
    & set(val_idx)
) == 0

assert len(
    set(train_idx)
    & set(test_idx)
) == 0

assert len(
    set(val_idx)
    & set(test_idx)
) == 0


if USE_SCAFFOLD_SPLIT:

    train_scaffolds = set(
        data.iloc[train_idx]["scaffold"]
    )

    val_scaffolds = set(
        data.iloc[val_idx]["scaffold"]
    )

    test_scaffolds = set(
        data.iloc[test_idx]["scaffold"]
    )

    print(
        "\nScaffold overlap train/val:",
        len(
            train_scaffolds
            & val_scaffolds
        )
    )

    print(
        "Scaffold overlap train/test:",
        len(
            train_scaffolds
            & test_scaffolds
        )
    )

    print(
        "Scaffold overlap val/test:",
        len(
            val_scaffolds
            & test_scaffolds
        )
    )


# ============================================================
# 7. SAVE SPLIT
# ============================================================

data["split"] = ""

data.loc[
    data.index[train_idx],
    "split"
] = "train"

data.loc[
    data.index[val_idx],
    "split"
] = "validation"

data.loc[
    data.index[test_idx],
    "split"
] = "test"

SPLIT_PATH = os.path.join(
    OUTPUT_DIR,
    "MOST_split.csv"
)

data.to_csv(
    SPLIT_PATH,
    index=False
)

print(
    "\nSplit сохранён:",
    SPLIT_PATH
)


# ============================================================
# 8. MOLECULAR DESCRIPTORS
#
# Это features для простого property baseline.
# ============================================================

print("\n" + "=" * 70)
print("5. FEATURE GENERATION")
print("=" * 70)

DESCRIPTOR_NAMES = [
    "MolWt",
    "LogP",
    "TPSA",
    "HBD",
    "HBA",
    "RotatableBonds",
    "RingCount",
    "FractionCSP3",
    "HeavyAtomCount",
    "AromaticRingCount"
]


def calculate_descriptors(mol):

    return np.array(
        [
            Descriptors.MolWt(mol),

            Crippen.MolLogP(mol),

            rdMolDescriptors.CalcTPSA(
                mol
            ),

            Lipinski.NumHDonors(
                mol
            ),

            Lipinski.NumHAcceptors(
                mol
            ),

            Lipinski.NumRotatableBonds(
                mol
            ),

            rdMolDescriptors.CalcNumRings(
                mol
            ),

            rdMolDescriptors.CalcFractionCSP3(
                mol
            ),

            Descriptors.HeavyAtomCount(
                mol
            ),

            rdMolDescriptors.CalcNumAromaticRings(
                mol
            )
        ],
        dtype=np.float32
    )


# ============================================================
# 9. MORGAN FP
# ============================================================

morgan = (
    rdFingerprintGenerator
    .GetMorganGenerator(
        radius=FP_RADIUS,
        fpSize=FP_SIZE
    )
)


X_desc = np.zeros(
    (
        len(data),
        len(DESCRIPTOR_NAMES)
    ),
    dtype=np.float32
)

X_fp = np.zeros(
    (
        len(data),
        FP_SIZE
    ),
    dtype=np.uint8
)

fingerprints = []

start = time.time()

for i, mol in enumerate(mols):

    X_desc[i] = (
        calculate_descriptors(
            mol
        )
    )

    fp = morgan.GetFingerprint(
        mol
    )

    fingerprints.append(fp)

    arr = np.zeros(
        FP_SIZE,
        dtype=np.uint8
    )

    DataStructs.ConvertToNumpyArray(
        fp,
        arr
    )

    X_fp[i] = arr


X_expert = np.hstack(
    [
        X_fp.astype(np.float32),
        X_desc
    ]
)


print(
    "Descriptor features:",
    X_desc.shape
)

print(
    "Expert features:",
    X_expert.shape
)

print(
    "Feature generation:",
    round(
        time.time() - start,
        1
    ),
    "sec"
)


# ============================================================
# 10. METRICS
# ============================================================

def metrics(
    y_true,
    y_pred
):

    return {
        "MAE":
            float(
                mean_absolute_error(
                    y_true,
                    y_pred
                )
            ),

        "RMSE":
            float(
                np.sqrt(
                    mean_squared_error(
                        y_true,
                        y_pred
                    )
                )
            ),

        "R2":
            float(
                r2_score(
                    y_true,
                    y_pred
                )
            )
    }


# ============================================================
# 11. PROPERTY BASELINE
#
# ВАЖНО:
# это НЕ B0 из исследовательского задания.
#
# Это простой baseline качества property prediction.
#
# B0 позже будет отдельным ГЕНЕРАТОРОМ.
# ============================================================

def make_property_baseline():

    return RandomForestRegressor(
        n_estimators=150,
        max_depth=14,
        min_samples_leaf=2,
        max_features=1.0,
        random_state=SEED,
        n_jobs=-1
    )


# ============================================================
# 12. EXPERT A
#
# ExtraTrees:
# Morgan fingerprints + descriptors
# ============================================================

def make_expert():

    return ExtraTreesRegressor(
        n_estimators=250,
        max_features="sqrt",
        min_samples_leaf=1,
        random_state=SEED,
        n_jobs=-1
    )


# ============================================================
# 13. TRAIN
# ============================================================

print("\n" + "=" * 70)
print("6. PROPERTY BASELINE + EXPERT A")
print("=" * 70)

results = []

baselines = {}
experts = {}


for target in TARGETS:

    print(
        "\nTARGET:",
        target
    )

    y = data[target].values.astype(
        np.float32
    )


    # --------------------------------------------------------
    # PROPERTY BASELINE
    # --------------------------------------------------------

    baseline = (
        make_property_baseline()
    )

    t0 = time.time()

    baseline.fit(
        X_desc[train_idx],
        y[train_idx]
    )

    pred_val = baseline.predict(
        X_desc[val_idx]
    )

    baseline_val_metrics = (
        metrics(
            y[val_idx],
            pred_val
        )
    )

    print(
        "Property baseline / validation:"
    )

    print(
        baseline_val_metrics
    )

    print(
        "Time:",
        round(
            time.time() - t0,
            1
        ),
        "sec"
    )

    results.append(
        {
            "target": target,
            "model": "property_baseline",
            "split": "validation",
            **baseline_val_metrics
        }
    )


    # --------------------------------------------------------
    # EXPERT A
    # --------------------------------------------------------

    expert = make_expert()

    t0 = time.time()

    expert.fit(
        X_expert[train_idx],
        y[train_idx]
    )

    pred_val = expert.predict(
        X_expert[val_idx]
    )

    expert_val_metrics = (
        metrics(
            y[val_idx],
            pred_val
        )
    )

    print(
        "Expert A / validation:"
    )

    print(
        expert_val_metrics
    )

    print(
        "Time:",
        round(
            time.time() - t0,
            1
        ),
        "sec"
    )

    baselines[target] = baseline
    experts[target] = expert

    results.append(
        {
            "target": target,
            "model": "expert_A",
            "split": "validation",
            **expert_val_metrics
        }
    )


# ============================================================
# 14. FINAL INTERNAL TEST
#
# Не переобучаемся на validation:
# основной train остаётся <30k.
# ============================================================

print("\n" + "=" * 70)
print("7. INTERNAL TEST")
print("=" * 70)


test_predictions = pd.DataFrame(
    {
        "comp_name":
            data.iloc[test_idx][
                "comp_name"
            ].values,

        "smiles":
            data.iloc[test_idx][
                "smiles"
            ].values
    }
)


for target in TARGETS:

    y = data[target].values.astype(
        np.float32
    )

    # baseline
    pred_baseline = (
        baselines[target]
        .predict(
            X_desc[test_idx]
        )
    )

    base_metrics = (
        metrics(
            y[test_idx],
            pred_baseline
        )
    )

    print(
        "\n",
        target,
        "- baseline test:"
    )

    print(
        base_metrics
    )

    results.append(
        {
            "target": target,
            "model": "property_baseline",
            "split": "test",
            **base_metrics
        }
    )


    # expert
    pred_expert = (
        experts[target]
        .predict(
            X_expert[test_idx]
        )
    )

    exp_metrics = (
        metrics(
            y[test_idx],
            pred_expert
        )
    )

    print(
        target,
        "- Expert A test:"
    )

    print(
        exp_metrics
    )

    results.append(
        {
            "target": target,
            "model": "expert_A",
            "split": "test",
            **exp_metrics
        }
    )

    test_predictions[
        f"true_{target}"
    ] = y[test_idx]

    test_predictions[
        f"pred_{target}"
    ] = pred_expert


# ============================================================
# 15. UNCERTAINTY
#
# Std predictions across ExtraTrees.
#
# Это ensemble uncertainty proxy,
# а НЕ строгий доверительный интервал.
# ============================================================

def tree_uncertainty(
    model,
    X
):

    predictions = np.vstack(
        [
            tree.predict(X)
            for tree
            in model.estimators_
        ]
    )

    mean = predictions.mean(
        axis=0
    )

    std = predictions.std(
        axis=0
    )

    return mean, std


for target in TARGETS:

    mean_pred, uncertainty = (
        tree_uncertainty(
            experts[target],
            X_expert[test_idx]
        )
    )

    test_predictions[
        f"uncertainty_{target}"
    ] = uncertainty


# ============================================================
# 16. APPLICABILITY DOMAIN
#
# Tanimoto similarity к фиксированной
# подвыборке TRAIN molecules.
#
# Threshold калибруется на VALIDATION,
# а TEST при этом не используется.
# ============================================================

print("\n" + "=" * 70)
print("8. APPLICABILITY DOMAIN")
print("=" * 70)

rng = np.random.default_rng(
    SEED
)

reference_size = min(
    AD_REFERENCE_SIZE,
    len(train_idx)
)

reference_indices = rng.choice(
    train_idx,
    size=reference_size,
    replace=False
)

reference_fps = [
    fingerprints[i]
    for i in reference_indices
]


def max_tanimoto(
    fp,
    reference
):

    sims = (
        DataStructs
        .BulkTanimotoSimilarity(
            fp,
            reference
        )
    )

    return float(
        max(sims)
    )


# ------------------------------------------------------------
# Калибровка threshold на VALIDATION
# ------------------------------------------------------------

val_similarity = []

t0 = time.time()

for idx in val_idx:

    val_similarity.append(
        max_tanimoto(
            fingerprints[idx],
            reference_fps
        )
    )


# 5% validation molecules допускается
# считать "вне домена".
AD_THRESHOLD = float(
    np.quantile(
        val_similarity,
        0.05
    )
)


print(
    "AD threshold:",
    round(
        AD_THRESHOLD,
        4
    )
)

print(
    "Median validation similarity:",
    round(
        float(
            np.median(
                val_similarity
            )
        ),
        4
    )
)

print(
    "AD calibration time:",
    round(
        time.time() - t0,
        1
    ),
    "sec"
)


# ------------------------------------------------------------
# Test AD
# ------------------------------------------------------------

test_similarity = []

for idx in test_idx:

    test_similarity.append(
        max_tanimoto(
            fingerprints[idx],
            reference_fps
        )
    )

test_predictions[
    "max_train_tanimoto"
] = test_similarity

test_predictions[
    "in_domain_A"
] = (
    test_predictions[
        "max_train_tanimoto"
    ]
    >= AD_THRESHOLD
)


print(
    "Test molecules in domain:",
    test_predictions[
        "in_domain_A"
    ].mean()
)


# ============================================================
# 17. RESULTS TABLE
# ============================================================

metrics_df = pd.DataFrame(
    results
)

print("\n" + "=" * 70)
print("9. METRICS")
print("=" * 70)

display(
    metrics_df.sort_values(
        [
            "target",
            "split",
            "model"
        ]
    )
)


# ============================================================
# 18. SAVE MODELS
# ============================================================

print("\n" + "=" * 70)
print("10. SAVE")
print("=" * 70)


for target in TARGETS:

    joblib.dump(
        experts[target],
        os.path.join(
            OUTPUT_DIR,
            f"expert_A_{target}.joblib"
        )
    )

    joblib.dump(
        baselines[target],
        os.path.join(
            OUTPUT_DIR,
            f"property_baseline_{target}.joblib"
        )
    )


# ============================================================
# 19. SAVE AD REFERENCE
# ============================================================

ad_reference_df = data.iloc[
    reference_indices
][
    [
        "comp_name",
        "smiles"
    ]
].copy()

ad_reference_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "AD_reference_A.csv"
    ),
    index=False
)


# ============================================================
# 20. SAVE TEST PREDICTIONS
# ============================================================

test_predictions.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Expert_A_test_predictions.csv"
    ),
    index=False
)


# ============================================================
# 21. SAVE METRICS
# ============================================================

metrics_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Expert_A_metrics.csv"
    ),
    index=False
)


# ============================================================
# 22. CONFIG
# ============================================================

config = {

    "seed":
        SEED,

    "dataset_size":
        int(len(data)),

    "train_size":
        int(len(train_idx)),

    "validation_size":
        int(len(val_idx)),

    "test_size":
        int(len(test_idx)),

    "split_type":
        (
            "Bemis-Murcko scaffold"
            if USE_SCAFFOLD_SPLIT
            else
            "target-stratified random"
        ),

    "targets":
        TARGETS,

    "fingerprint": {
        "type":
            "Morgan",

        "radius":
            FP_RADIUS,

        "n_bits":
            FP_SIZE
    },

    "descriptors":
        DESCRIPTOR_NAMES,

    "property_baseline":
        (
            "RandomForestRegressor "
            "on molecular descriptors"
        ),

    "expert_A":
        (
            "ExtraTreesRegressor "
            "on Morgan fingerprints "
            "+ descriptors"
        ),

    "uncertainty":
        (
            "standard deviation "
            "between ExtraTrees "
            "estimators"
        ),

    "AD_method":
        (
            "maximum Morgan Tanimoto "
            "similarity to fixed "
            "training reference subset"
        ),

    "AD_reference_size":
        int(reference_size),

    "AD_threshold":
        AD_THRESHOLD
}


with open(
    os.path.join(
        OUTPUT_DIR,
        "Expert_A_config.json"
    ),
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 23. PREDICTION FUNCTION
#
# Позже этой функцией будем оценивать
# молекулы от B0 и M1.
# ============================================================

def predict_expert_A(
    smiles_list
):

    new_mols = []

    valid_mask = []

    for smi in smiles_list:

        mol = Chem.MolFromSmiles(
            smi
        )

        new_mols.append(
            mol
        )

        valid_mask.append(
            mol is not None
        )


    result = pd.DataFrame(
        {
            "smiles":
                smiles_list,

            "valid":
                valid_mask
        }
    )


    valid_indices = [
        i
        for i, ok
        in enumerate(valid_mask)
        if ok
    ]


    if len(valid_indices) == 0:
        return result


    n = len(valid_indices)


    Xd = np.zeros(
        (
            n,
            len(DESCRIPTOR_NAMES)
        ),
        dtype=np.float32
    )

    Xf = np.zeros(
        (
            n,
            FP_SIZE
        ),
        dtype=np.uint8
    )

    fps_new = []


    for j, original_i in enumerate(
        valid_indices
    ):

        mol = new_mols[
            original_i
        ]

        Xd[j] = (
            calculate_descriptors(
                mol
            )
        )

        fp = morgan.GetFingerprint(
            mol
        )

        fps_new.append(
            fp
        )

        arr = np.zeros(
            FP_SIZE,
            dtype=np.uint8
        )

        DataStructs.ConvertToNumpyArray(
            fp,
            arr
        )

        Xf[j] = arr


    Xe = np.hstack(
        [
            Xf.astype(
                np.float32
            ),
            Xd
        ]
    )


    for target in TARGETS:

        mean_pred, unc = (
            tree_uncertainty(
                experts[target],
                Xe
            )
        )

        result.loc[
            valid_indices,
            f"pred_{target}"
        ] = mean_pred

        result.loc[
            valid_indices,
            f"uncertainty_{target}"
        ] = unc


    similarities = [
        max_tanimoto(
            fp,
            reference_fps
        )
        for fp in fps_new
    ]

    result.loc[
        valid_indices,
        "max_train_tanimoto_A"
    ] = similarities

    result.loc[
        valid_indices,
        "in_domain_A"
    ] = (
        np.asarray(
            similarities
        )
        >= AD_THRESHOLD
    )

    return result


# ============================================================
# 24. QUICK CHECK
# ============================================================

print("\n" + "=" * 70)
print("11. QUICK CHECK")
print("=" * 70)

example_smiles = (
    data.iloc[
        test_idx[:5]
    ]["smiles"]
    .tolist()
)

display(
    predict_expert_A(
        example_smiles
    )
)


# ============================================================
# 25. FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("DATABASE A / EXPERT A ГОТОВ")
print("=" * 70)

print(
    "\nDATA:"
)

print(
    len(data),
    "молекул"
)

print(
    "\nSPLIT:"
)

print(
    "train      =",
    len(train_idx)
)

print(
    "validation =",
    len(val_idx)
)

print(
    "test       =",
    len(test_idx)
)

print(
    "\nPROPERTY BASELINE:"
)

print(
    "Random Forest + descriptors"
)

print(
    "\nEXPERT A:"
)

print(
    "ExtraTrees + Morgan "
    "+ descriptors"
)

print(
    "\nTARGETS:"
)

print(
    "energy_density_MJ_kg"
)

print(
    "tbr_kJ_mol"
)

print(
    "\nUNCERTAINTY:"
)

print(
    "std между деревьями"
)

print(
    "\nAD threshold:",
    AD_THRESHOLD
)

print(
    "\nРезультаты сохранены в:"
)

print(
    OUTPUT_DIR
)

1. LOAD DATABASE A
Размер: (33706, 4)
Уникальных SMILES: 33706


,comp_name,smiles,energy_density_MJ_kg,tbr_kJ_mol
0,A-0_A-0_A-0_A-0,C1=CC2C=CC1C2,0.088030,320.298085
1,A-1_A-0_A-0_A-0,FC1=CC2C=CC1C2,0.203054,290.434249
2,A-2_A-0_A-0_A-0,FC(F)(F)C1=CC2C=CC1C2,0.047062,284.936736
3,A-3_A-0_A-0_A-0,N#CC1=CC2C=CC1C2,0.083948,258.947101
4,A-4_A-0_A-0_A-0,O=[N+]([O-])C1=CC2C=CC1C2,0.039601,219.128675



2. SMILES VALIDATION
Валидных: 33706
Невалидных: 0

3. SCAFFOLD DIAGNOSTICS
Уникальных scaffold: 89
Самый большой scaffold: 8380 молекул
Доля крупнейшего scaffold: 0.2486

Выбранный split: scaffold

4. SPLIT RESULT
Train: 26900 (0.798)
Validation: 4089 (0.121)
Test: 2717 (0.081)

Scaffold overlap train/val: 0
Scaffold overlap train/test: 0
Scaffold overlap val/test: 0

Split сохранён: /content/MOST_expert_A/MOST_split.csv

5. FEATURE GENERATION
Descriptor features: (33706, 10)
Expert features: (33706, 1034)
Feature generation: 23.5 sec

6. PROPERTY BASELINE + EXPERT A

TARGET: energy_density_MJ_kg
Property baseline / validation:
{'MAE': 0.007583955082839348, 'RMSE': 0.010484391862450437, 'R2': 0.3117435750626739}
Time: 4.7 sec
Expert A / validation:
{'MAE': 0.005607323916003064, 'RMSE': 0.009263984979657741, 'R2': 0.46264725795342276}
Time: 38.1 sec

TARGET: tbr_kJ_mol
Property baseline / validation:
{'MAE': 12.08792556775605, 'RMSE': 15.742574646838408, 'R2': -0.32504330140472115}
Ti

,target,model,split,MAE,RMSE,R2
5,energy_density_MJ_kg,expert_A,test,0.007784,0.011369,0.406458
4,energy_density_MJ_kg,property_baseline,test,0.014667,0.016978,-0.323773
1,energy_density_MJ_kg,expert_A,validation,0.005607,0.009264,0.462647
0,energy_density_MJ_kg,property_baseline,validation,0.007584,0.010484,0.311744
7,tbr_kJ_mol,expert_A,test,7.971000,10.022278,0.459595
6,tbr_kJ_mol,property_baseline,test,11.909501,15.269297,-0.254367
3,tbr_kJ_mol,expert_A,validation,11.015902,13.937097,-0.038540
2,tbr_kJ_mol,property_baseline,validation,12.087926,15.742575,-0.325043



10. SAVE

11. QUICK CHECK


,smiles,valid,pred_energy_density_MJ_kg,uncertainty_energy_density_MJ_kg,pred_tbr_kJ_mol,uncertainty_tbr_kJ_mol,max_train_tanimoto_A,in_domain_A
0,C[S+]=C1C=CC(C2=CC3C=CC2C3)=C([N+](=O)[O-])C1=...,True,0.052973,0.010106,158.772955,53.519203,0.714286,True
1,C[S+]=C1C(=[NH2+])C([N+](=O)[O-])=CC=C1C1=CC2C...,True,0.069101,0.013489,232.476917,20.732751,0.733333,True
2,C[S+]=C1C(C2=CC3C=CC2C3)=CC=C([N+](=O)[O-])C1=...,True,0.053378,0.011967,223.900557,15.918968,0.666667,True
3,N#CC1=C(C#CC2=CC3C=CC2C3)C=CC(=[N+]([O-])[O-])...,True,0.056428,0.026575,222.819437,20.394658,0.534483,False
4,C[N+](C)=C1C=CC(C#CC2=CC3C=CC2C3)=C([N+](=O)[O...,True,0.060382,0.038150,212.812663,16.851271,0.716981,True



DATABASE A / EXPERT A ГОТОВ

DATA:
33706 молекул

SPLIT:
train      = 26900
validation = 4089
test       = 2717

PROPERTY BASELINE:
Random Forest + descriptors

EXPERT A:
ExtraTrees + Morgan + descriptors

TARGETS:
energy_density_MJ_kg
tbr_kJ_mol

UNCERTAINTY:
std между деревьями

AD threshold: 0.62

Результаты сохранены в:
/content/MOST_expert_A
